# EchoClause — Build with Gemma Hackathon Demo

Evidence-grounded promise-to-contract reconciliation using **Gemma 4** multimodal extraction and deterministic calculators.

> EchoClause compares representations across supplied evidence. It does not provide legal advice or determine legal enforceability.

In [ ]:
# Cell 1 — Configuration
RUN_FULL_BENCHMARK = False  # Keep False for hackathon demo
PROJECT_DIR = "/kaggle/working"
MODEL_PRIMARY = "google/gemma-4-E4B-it"
MODEL_FALLBACK = "google/gemma-4-E2B-it"

In [ ]:
# Cell 2 — Copy source dataset and install
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

INPUT = Path("/kaggle/input/echo-clause-gemma4-src")
DEST = Path("/kaggle/working/echo-clause-gemma4")
if DEST.exists():
    shutil.rmtree(DEST)
DEST.mkdir(parents=True)

if (INPUT / "pyproject.toml").exists() and not list(INPUT.glob("*.zip")):
    shutil.copytree(INPUT, DEST, dirs_exist_ok=True)
else:
    for item in INPUT.iterdir():
        if item.suffix == ".zip":
            with zipfile.ZipFile(item) as zf:
                zf.extractall(DEST)
        elif item.is_file():
            shutil.copy2(item, DEST / item.name)

if not (DEST / "pyproject.toml").exists():
    raise FileNotFoundError(f"pyproject.toml missing after staging. INPUT={list(INPUT.iterdir())}")

os.chdir(DEST)
for k in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY"):
    os.environ.pop(k, None)

# Keep Kaggle preinstalled torch; upgrade only Gemma stack + project deps
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=5.14", "accelerate", "bitsandbytes", "sentencepiece",
    "protobuf", "httpx[socks]", "pydantic", "pillow", "pytest", "ruff",
])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
from echo_clause.config import kaggle_models_available
print("Kaggle Gemma models mounted:", kaggle_models_available())
print("HF_TOKEN set (optional):", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# Cell 3 — Generate demo assets
import subprocess, sys
subprocess.check_call([sys.executable, "scripts/generate_demo_assets.py"])

In [ ]:
# Cell 4 — R1 Runtime spike (Gemma multimodal + function calling)
import json
import os
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(DEST))
from echo_clause.config import kaggle_models_available

spike_cmd = [sys.executable, "scripts/run_runtime_spike.py"]
if not kaggle_models_available() and not os.environ.get("HF_TOKEN"):
    spike_cmd.append("--not-run-gpu")
    print("No Kaggle model mount and no HF_TOKEN — writing NOT_RUN_GPU artifact")
else:
    print("Live Gemma spike enabled via Kaggle Models or HF_TOKEN")

result = subprocess.run(spike_cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

runs = sorted(Path("artifacts/runs").glob("runtime_spike_*.json"))
assert runs, "Missing runtime spike artifact"
artifact = json.loads(runs[-1].read_text())
print("Spike status:", artifact.get("status"))
if artifact.get("status") == "PASSED":
    print("R1 live Gemma spike PASSED")
else:
    print("R1 partial — accept license at kaggle.com/models/google/gemma-4")

In [ ]:
# Cell 4b — Multimodal extraction API (code path; live when runtime.load() succeeds)
import json
from pathlib import Path

from echo_clause.config import ASSETS_DIR
from echo_clause.gemma_runtime import GemmaRuntime
from echo_clause.schemas import SourceType

# Live Gemma paths (see echo_clause/gemma_runtime.py):
#   runtime.extract_claims_from_image(ASSETS_DIR / "advertisement.png", "advertisement", SourceType.ADVERTISEMENT)
#   runtime.extract_claims_from_image(ASSETS_DIR / "support_chat.png", "support_chat", SourceType.SUPPORT_CHAT)
#   runtime.extract_claims_from_image(ASSETS_DIR / "contract.png", "contract", SourceType.CONTRACT)
#   runtime.extract_claims_from_audio(ASSETS_DIR / "sales_pitch.wav", "sales_pitch")
#   runtime.run_function_call_demo()  # allowlisted calculate_fee_percentage, etc.

rec = json.loads((ASSETS_DIR / "recorded_claims.json").read_text(encoding="utf-8"))
by_type: dict[str, int] = {}
for c in rec["claims"]:
    by_type[c["source_type"]] = by_type.get(c["source_type"], 0) + 1
print("Recorded replay fixture:", rec.get("model_id"), "—", len(rec["claims"]), "claims")
for st, n in sorted(by_type.items()):
    print(f"  {st}: {n} claims")
print("When GPU load fails, Cell 5 uses this fixture (not synthetic placeholders).")

In [ ]:
# Cell 5 — Full pipeline (live Gemma extraction)
import sys
from pathlib import Path

DEST = Path("/kaggle/working/echo-clause-gemma4")
sys.path.insert(0, str(DEST))
from echo_clause.gemma_runtime import GemmaRuntime
from echo_clause.pipeline import run_pipeline, write_pipeline_artifact

runtime = GemmaRuntime()
loaded = runtime.load()
if not loaded:
    print("Live load failed — using recorded replay for pipeline validation")
    report = run_pipeline(use_recorded=True)
else:
    report = run_pipeline(runtime=runtime, audio_fallback=True)
artifact = write_pipeline_artifact(report, prefix="kaggle_pipeline")
print(f"Model: {report.get('model_id')}")
print(f"Conflicts: {report['conflict_count']}")
print(f"Gold: {report['demo_validation']}")
print(f"Artifact: {artifact}")

In [ ]:
# Cell 6 — Validate against gold.json (must detect 5/5 contradictions)
validation = report["demo_validation"]
assert validation["all_gold_detected"], f"Missing: {validation.get('missing_fields')}"
assert validation["detected_gold_contradictions"] >= 5
print("PASS: 5/5 gold contradictions detected")

In [ ]:
# Cell 7 — Unit tests (skip extended benchmark unless enabled)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pytest", "-q", "--ignore=tests/test_benchmark.py" if not RUN_FULL_BENCHMARK else "-q"])
print("All tests passed")

## Results

EchoClause extracted claims from advertisement, sales audio, support chat, and contract images, normalized financial terms deterministically, and flagged contradictions before signing.